In [9]:
import pandas as pd
import numpy as np
import hnswlib as hl
import hnsw_benchmark2 as bm
import matplotlib.pyplot as plt
import os

In [10]:
dataset_name = "arguana"
corpus_filename = dataset_name + "_corpus_lid"
corpus_input_path = os.path.join("..\\data\\", corpus_filename + ".json")
df_corpus = pd.read_json(corpus_input_path, orient="records", lines=True)

query_filename = dataset_name + "_query"
query_input_path = os.path.join("..\\data\\", query_filename + ".json")
df_query = pd.read_json(query_input_path, orient="records", lines=True)

In [11]:
print("Corpus data")
# print(df_corpus)
# print(df_corpus.keys)
corpus_vectors = np.asarray([np.asarray(v) for v in df_corpus.vectors], dtype=np.float32)
print(corpus_vectors.shape)

print("Query data")
# print(df_query)
# print(df_query.keys)
query_vectors = np.asarray([np.asarray(v) for v in df_query.vectors], dtype=np.float32)
print(query_vectors.shape)


Corpus data
(8674, 384)
Query data
(1406, 384)


In [12]:
# KNN számítás minden query vectorra - TESZT
k = 10
scores = query_vectors @ corpus_vectors.T

topk = np.argpartition(
    -scores,
    kth=k - 1,
    axis=1,
)[:, :k]

# Optional: sort those k results by actual score
rows = np.arange(len(df_query.vectors))[:, None]
order = np.argsort(-scores[rows, topk], axis=1)

knn_matrix = np.take_along_axis(topk, order, axis=1)

print(knn_matrix.shape)
# for i in knn_matrix[0]:
#     print(f"{i:4d} - dist: {hnswbm.KNN.euclidian_distance(df_query.vectors[0], df_corpus.vectors[i]):.4f}")

(1406, 10)


In [13]:
# HNSW konstruálás - TESZT
hnsw = hl.Index(space             = 'ip',
                dim               = corpus_vectors.shape[1])
hnsw.init_index(max_elements      = corpus_vectors.shape[0],
                M                 = 16,
                ef_construction   = 128)
hnsw.add_items(data               = corpus_vectors)

In [14]:
# HNSW futtatás - TESZT
hnsw.set_ef(10)

hnsw_indicies, _ = hnsw.knn_query(query_vectors, k=10)

print(hnsw_indicies.shape)

(1406, 10)


In [15]:
# Recall számítás - TESZT
recalls = []

for R_A, R_E in zip(hnsw_indicies, knn_matrix):
    recalls.append(
        len(set(R_A) & set(R_E)) / len(R_E)
    )

print(np.min(recalls))
print(np.mean(recalls))
print(np.max(recalls))

0.0
0.919914651493599
1.0


In [16]:
# HNSW benchmark
bm_random = bm.HNSW_benchmark(data=df_corpus.vectors)
recalls_random = bm_random.run(qs=df_query.vectors, ef_search=10)

print(f"Minimum recall: {np.min(recalls_random):.4f}")
print(f"Average recall: {np.mean(recalls_random):.4f}")
print(f"Maximum recall: {np.max(recalls_random):.4f}")

Minimum recall: 0.3000
Average recall: 0.9184
Maximum recall: 1.0000


In [ ]:
# HNSW benchmark - LID desc
lid_desc_indices = np.argsort(df_corpus.LID)
bm_random = bm.HNSW_benchmark(data=[df_corpus.vector[i] for i in lid_desc_indices])
recalls_random = bm_random.run(qs=df_query.vectors, ef_search=10)

print(f"Minimum recall: {np.min(recalls_random):.4f}")
print(f"Average recall: {np.mean(recalls_random):.4f}")
print(f"Maximum recall: {np.max(recalls_random):.4f}")

In [ ]:
# LID :)
df_lid = bm.calculate_lid(df_corpus.vectors)

print(df_lid[:10])

In [ ]:
# LID - TESZT
# KNN számítás minden pontra
k = 100
data = np.asarray(df_corpus.vectors.tolist(), dtype=np.float32)
scores = data @ data.T

# Assign the same ID to identical vectors, then exclude them from each row.
_, vector_ids = np.unique(data, axis=0, return_inverse=True)
for row, vector_id in enumerate(vector_ids):
    scores[row, vector_ids == vector_id] = -np.inf

topk = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]

rows = np.arange(len(data))[:, None]
order = np.argsort(-scores[rows, topk], axis=1)

knn_matrix = np.take_along_axis(topk, order, axis=1)

# print(knn_matrix[14][:10])
print(knn_matrix[14][-10:])
# for i in knn_matrix[14][:10]:
for i in knn_matrix[14][-10:]:
    print(bm.euclidian_distance(corpus_vectors[i], corpus_vectors[14]))


knn = bm.KNN(corpus_vectors)
lid = knn.get_KNN(corpus_vectors[14], k)
# print(lid[:10])
print(lid[-10:])
# for i in lid[:10]:
for i in lid[-10:]:
    print(bm.euclidian_distance(corpus_vectors[i], corpus_vectors[14]))